In [1]:
import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

from sales_dataset import SalesDataset

In [2]:
# df_xls = pd.read_excel('temp_8_9_10_итог_с_июлем.xlsx')
# ds = SalesDataset(df_xls, start_date='2024-01-01', cn_mes=19)
# X, y = ds.load_data_regress()

df_csv = pd.read_csv('features_sale.csv')
y = df_csv[['fact_sale']].to_numpy().reshape(-1)
df_x = df_csv.drop(['fact_sale'], axis=1)
X = df_x.to_numpy()

In [3]:
# 2. Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Создание датасетов LightGBM
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [4]:
# Оценка качества модели
def evaluate_model(y_true, y_pred, model_name):
    """Функция для оценки модели регрессии"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{model_name} Результаты:")
    print(f"Среднеквадратичная ошибка (MSE): {mse:.4f}")
    print(f"Корень из MSE (RMSE): {rmse:.4f}")
    print(f"Средняя абсолютная ошибка (MAE): {mae:.4f}")
    print(f"Коэффициент детерминации (R²): {r2:.4f}")

    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

**Пример использования ранней остановки**

In [9]:
# Параметры модели
params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.01,
    'max_depth': 5,
    'num_leaves': 31,
    'verbosity': -1
}

# Обучение с ранней остановкой
model_early_stop = lgb.train(
    params,
    train_data,
    valid_sets=[valid_data],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=300), # Если в течение 50 итераций подряд метрика не стала лучше
        lgb.log_evaluation(50)  # Выводим логи каждые 50 итераций
    ]
)

# Сохранение / Загрузка модели
model_early_stop.save_model("lgb_model_regress.txt")
loaded_model = lgb.Booster(model_file="lgb_model_regress.txt")

# Предсказание на тестовых данных
y_pred = loaded_model.predict(X_test)
early_results = evaluate_model(y_test, y_pred, "LightGBM с ранней остановкой")

print(f"\nКоличество использованных деревьев: {model_early_stop.best_iteration}")

Training until validation scores don't improve for 300 rounds
[50]	valid_0's l1: 5.60159
[100]	valid_0's l1: 4.7356
[150]	valid_0's l1: 4.37886
[200]	valid_0's l1: 4.25569
[250]	valid_0's l1: 4.2168
[300]	valid_0's l1: 4.2096
[350]	valid_0's l1: 4.21863
[400]	valid_0's l1: 4.21177
[450]	valid_0's l1: 4.20215
[500]	valid_0's l1: 4.20032
[550]	valid_0's l1: 4.20101
[600]	valid_0's l1: 4.20068
[650]	valid_0's l1: 4.20145
[700]	valid_0's l1: 4.20291
[750]	valid_0's l1: 4.20381
[800]	valid_0's l1: 4.20476
[850]	valid_0's l1: 4.20583
[900]	valid_0's l1: 4.20829
Early stopping, best iteration is:
[609]	valid_0's l1: 4.20013

LightGBM с ранней остановкой Результаты:
Среднеквадратичная ошибка (MSE): 39.2743
Корень из MSE (RMSE): 6.2669
Средняя абсолютная ошибка (MAE): 4.2001
Коэффициент детерминации (R²): 0.6528

Количество использованных деревьев: 609


**Оптимизация гиперпараметров с помощью GridSearchCV**

In [6]:
# Определяем сетку параметров для поиска
param_grid = {
    'n_estimators': [100, 200, 300], # Количество деревьев
    'learning_rate': [0.01, 0.1, 0.2], # Скорость обучения
    'max_depth': [3, 5, 7, 9, 10, 12], # Максимальная глубина деревьев
    'num_leaves': [31, 50, 100], # Количество листьев в дереве
    'subsample': [0.5, 0.8, 1.0] # Доля выборки для обучения каждого дерева
}

# Создаем и обучаем модель с поиском по сетке
grid_search = GridSearchCV(
    estimator=lgb.LGBMRegressor(random_state=42, verbosity=-1),
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=0
)

In [7]:
grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший MSE: {-grid_search.best_score_:.4f}")

# Используем лучшую модель
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
best_results = evaluate_model(y_test, y_pred_best, "Оптимизированный LightGBM")

Лучшие параметры: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'num_leaves': 31, 'subsample': 0.5}
Лучший MSE: 38.1696

Оптимизированный LightGBM Результаты:
Среднеквадратичная ошибка (MSE): 39.6891
Корень из MSE (RMSE): 6.2999
Средняя абсолютная ошибка (MAE): 4.2334
Коэффициент детерминации (R²): 0.6492
